In [ ]:
# Setup (added for the release): the notebook's folder on sys.path, then work in its data folder.
# The curation ran with the current directory = that folder; see curation/README.md for its contents.
import os, sys
if "data/curation/" not in os.getcwd().replace(os.sep, "/") + "/":
    sys.path.insert(0, os.path.abspath("."))
    os.chdir('../../data/curation/rBMRB')
print("working folder:", os.getcwd())


# Assigned chemical-shift lists, then pH and temperature

## One record per assigned shift list

In [ ]:
import sys
MAKESHIFT_ROOT = "../../../external/makeshift"
if MAKESHIFT_ROOT not in sys.path:
    sys.path.insert(0, MAKESHIFT_ROOT)

import makeshift as ms
import os
import glob
import tqdm
import pickle

def get_chem_shifts(entry):
    """[MIGRATED] replaces parser.py's get_chem_shifts with makeshift's
    ChemicalShifts; keeps the cs_saveframe_id column name so downstream
    grouping/filtering code is unchanged."""
    return ms.ChemicalShifts.from_entry(entry).data.rename(columns={"ChemShift_ID": "cs_saveframe_id"})

_NULL_VALUES = {'.', '', 'none', 'nan', 'na', 'n/a', 'n.a'}

def _parse_ph(raw):
    """Robustly parse a pH value to float, or None if missing/unparseable."""
    if raw is None:
        return None
    s = str(raw).strip()
    if s.lower() in _NULL_VALUES:
        return None
    try:
        return float(s)
    except (ValueError, TypeError):
        return None

def _parse_temperature(raw):
    """Robustly parse a temperature value to float (Kelvin), or None if
    missing/unparseable. Strips a trailing K/k unit suffix (e.g. '298K')."""
    if raw is None:
        return None
    s = str(raw).strip()
    if s.lower() in _NULL_VALUES:
        return None
    if 'k' in s.lower():
        s = s.lower().replace('k', '').strip()
    try:
        return float(s)
    except (ValueError, TypeError):
        return None

def _resolve_label(label, all_framecodes):
    """Resolve a BMRB '.'/'$framecode' style cross-reference label to the
    actual framecode(s) it points to. '.' means "unspecified" -- fall back to
    every framecode available in that category (usually just one); otherwise
    strip the leading '$'. [ROBUSTNESS] uses split("$")[-1] rather than [1]
    so a label with no '$' at all (malformed deposit) returns itself instead
    of raising IndexError and killing the whole extraction loop."""
    if label == '.':
        return list(all_framecodes)
    return [label.split("$")[-1]]

base_path = os.getcwd()
data_path = os.path.join(base_path, "bmrb_entries")
result_dict = {}
failed_entries = []

for name in tqdm.tqdm(os.listdir(data_path)):
    if name.startswith('.'):
        continue

    entry_id = name.split("bmr")[1]
    folder_path = os.path.join(data_path, name)

    if os.path.isdir(folder_path):
        str_files = glob.glob(os.path.join(folder_path, "*_3.str"))
        if not str_files:
            print(f"No *_3.str file found in {folder_path}, skipping.")
            failed_entries.append(name)
            continue

        # [ROBUSTNESS] a single malformed entry (unparseable .str, or one
        # with no assigned_chemical_shifts at all -- get_chem_shifts raises
        # ValueError in that case) used to kill this whole ~10k-entry loop
        # with no partial save. Log and skip instead.
        try:
            entry = ms.NMRStarEntry.from_file(str_files[0])
            df = get_chem_shifts(entry)
        except Exception as e:
            print(f"Failed to read {name} ({type(e).__name__}: {e}), skipping.")
            failed_entries.append(name)
            continue

        "Some entries have several cs_saveframe_id, instead of averaging them, we will just note them and process them later"
        entry_data = entry.data

        # [MAKESHIFT] resolved once per entry (not per assignment) since they
        # don't vary across an entry's assignment experiments.
        entity_frames = entry.saveframe("entity")
        first_entity_fc = next(iter(entity_frames), None)
        first_entity_id = entity_frames[first_entity_fc].get("ID") if first_entity_fc else None
        ref_df = entry.shift_reference()
        cond_df = entry.sample_conditions()

        assign_index = 1
        for assign_name in entry_data['assigned_chemical_shifts'].keys():
            entry_key = f"{name}_{assign_index}"
            try:
                chem_shift_exp = entry_data['assigned_chemical_shifts'][assign_name]["_Chem_shift_experiment"]
                all_labels = [item['Sample_label'] for item in chem_shift_exp]
            except KeyError:
                print(f"Chem_shift_experiment is missing for {assign_name} in {name}, skipping.")
                all_labels = []

            # [MAKESHIFT] entry.is_deuterated(sample_name=..., entity_id=...)
            # replaces the manual entry_data['sample'][s]['_Sample_component']
            # + Entity_label substring walk: it's built on makeshift's own
            # sample_info() (graceful on missing loops/columns) and checks the
            # same deuteration keywords ('2h'/'deuter') as the "Screen
            # deuteration" cell downstream, so both stages agree by construction.
            sample_names = []
            for label in all_labels:
                sample_names.extend(_resolve_label(label, entry_data.get('sample', {}).keys()))

            deuterated_flags = []
            for s in sample_names:
                try:
                    deuterated_flags.append(bool(entry.is_deuterated(sample_name=s, entity_id=first_entity_id)))
                except Exception as e:
                    print(f"Some deuteration information is missing for {s} in {name} ({type(e).__name__}: {e})")

            result_dict[entry_key] = {}
            # [FIX] gate on deuterated_flags, not sample_names: if every
            # is_deuterated() call above raised, deuterated_flags stays empty
            # even though sample_names was non-empty -- any([]) is False, which
            # used to silently record "confirmed not deuterated" for an entry
            # we actually failed to check. None ("unknown") is the correct
            # result whenever we got zero successful readings, regardless of
            # whether that's because there were no samples to check or because
            # every check on existing samples errored out.
            result_dict[entry_key]['deuterated'] = any(deuterated_flags) if deuterated_flags else None

            # [MAKESHIFT] entry.shift_reference() replaces the manual
            # entry_data['chem_shift_reference'][tag]['_Chem_shift_ref'] walk.
            reference_tag = entry_data['assigned_chemical_shifts'][assign_name].get('Chem_shift_reference_label', '.')
            ref_candidates = _resolve_label(reference_tag, ref_df['reference'].unique() if not ref_df.empty else [])
            # [ROBUSTNESS] original code used `assert len(...) == 1` here,
            # which crashes the *entire* extraction loop the moment one entry
            # has multiple chem_shift_reference framecodes with an unlabeled
            # '.' Chem_shift_reference_label. Fall back to None instead.
            if reference_tag == '.' and len(ref_candidates) != 1:
                print(f"Chem_shift_reference is ambiguous or missing for {assign_name} in {name}, skipping.")
                result_dict[entry_key]['reference'] = None
            else:
                rows = ref_df[ref_df['reference'].isin(ref_candidates)] if not ref_df.empty else ref_df
                names = sorted(set(rows['Mol_common_name'].dropna())) if not rows.empty else []
                result_dict[entry_key]['reference'] = names or None

            # [MAKESHIFT] entry.sample_conditions() replaces the manual
            # _Sample_condition_variable loop; pH/temperature still go through
            # _parse_ph/_parse_temperature since makeshift's own numeric
            # coercion leaves some null markers ('na') and K-suffixed values
            # ('298K') as raw, unconverted strings.
            condition_name = entry_data['assigned_chemical_shifts'][assign_name].get("Sample_condition_list_label", '.')
            cond_candidates = _resolve_label(condition_name, cond_df['sample_conditions'].unique() if not cond_df.empty else [])
            if condition_name == '.' and len(cond_candidates) != 1:
                ph_raw = temp_raw = None
            else:
                row = cond_df[cond_df['sample_conditions'].isin(cond_candidates)]
                ph_raw = row['pH'].iloc[0] if 'pH' in row.columns and len(row) else None
                temp_raw = row['temperature'].iloc[0] if 'temperature' in row.columns and len(row) else None

            result_dict[entry_key]['pH'] = _parse_ph(ph_raw)
            result_dict[entry_key]['temperature'] = _parse_temperature(temp_raw)

            result_dict[entry_key]['chemical shifts'] = df[df['cs_saveframe_id'] == assign_name]
            assign_index += 1

print(f"Skipped {len(failed_entries)} entries that could not be read: {failed_entries}")

with open(os.path.join(base_path, "bmrb_cs_data_raw_beforestage2.pkl"), 'wb') as f:
    pickle.dump(result_dict, f)


In [ ]:
import pickle
import os

base_path = os.getcwd()
with open(os.path.join(base_path, "bmrb_cs_data_raw_beforestage2.pkl"), 'rb') as f:
    result_dict = pickle.load(f)

print(len(result_dict.keys()))

## pH outside 4.5-8.5

In [ ]:
import pickle
import os
import tqdm

base_path = os.getcwd()
with open(os.path.join(base_path, "bmrb_cs_data_raw_beforestage2.pkl"), 'rb') as f:
    result_dict = pickle.load(f)

badph = 0
real_badph = 0
real_badph_entries = []
real_badph_entries_noinfo = []

# [ROBUSTNESS] pH was already parsed to a clean float (or None) at
# extraction time in cell 1 via _parse_ph -- no float()/try-except needed
# here, and no risk of a raw string like 'na' silently surviving as "valid".
for name in tqdm.tqdm(list(result_dict.keys())):
    ph_value = result_dict[name]['pH']
    if ph_value is None:
        del result_dict[name]
        badph += 1
        real_badph_entries_noinfo.append(name)
        continue

    if ph_value < 4.5 or ph_value > 8.5:
        del result_dict[name]
        badph += 1
        real_badph += 1
        real_badph_entries.append(name)

print(f"Removed {badph} entries due to pH issues or unclear info.")
print(f"Among them, {real_badph} entries have confirmed pH values outside the acceptable range.")

with open(os.path.join(base_path, "bmrb_cs_data_raw_stage2_noph.pkl"), 'wb') as f:
    pickle.dump(result_dict, f)


In [ ]:
with open(os.path.join(base_path, "bmrb_cs_data_raw_stage2_noph.pkl"), 'rb') as f:
    result_dict = pickle.load(f)
print(len(result_dict.keys()))

## Temperature outside 293-313 K

In [ ]:
import pickle
import os
import tqdm

base_path = os.getcwd()
with open(os.path.join(base_path, "bmrb_cs_data_raw_stage2_noph.pkl"), 'rb') as f:
    result_dict = pickle.load(f)

badtemp = 0
real_badtemp = 0
real_badtemp_entries = []
real_badtemp_entries_noinfo = []

# [ROBUSTNESS] temperature was already parsed to a clean float in Kelvin (or
# None) at extraction time in cell 1 via _parse_temperature, which also
# strips a trailing K/k unit suffix (e.g. '298K') -- no re-parsing here.
for name in tqdm.tqdm(list(result_dict.keys())):
    temp_value = result_dict[name]['temperature']
    if temp_value is None:
        del result_dict[name]
        badtemp += 1
        real_badtemp_entries_noinfo.append(name)
        continue

    if temp_value < 293 or temp_value > 313:
        del result_dict[name]
        badtemp += 1
        real_badtemp += 1
        real_badtemp_entries.append(name)

print(f"Removed {badtemp} entries due to temperature issues or unclear info.")
print(f"Among them, {real_badtemp} entries have confirmed temperature values outside the acceptable range.")

with open(os.path.join(base_path, "bmrb_cs_data_raw_stage2_noph_notemp.pkl"), 'wb') as f:
    pickle.dump(result_dict, f)

In [ ]:
with open(os.path.join(base_path, "bmrb_cs_data_raw_stage2_noph_notemp.pkl"), 'rb') as f:
    result_dict = pickle.load(f)
print(len(result_dict.keys()))